# 02 -- Sentiment & Emotion Analysis

Scores every tweet with two complementary sentiment systems and a full
Ekman-emotion classifier:

| Model | What it adds |
|---|---|
| **VADER** | Rule-based, fast; neg / neu / pos / compound scores |
| **RoBERTa** (`cardiffnlp/twitter-roberta-base-sentiment-latest`) | Fine-tuned on ~124 M tweets; neg / neu / pos probabilities + compound |
| **DistilRoBERTa** (`j-hartmann/emotion-english-distilroberta-base`) | 7-class Ekman emotions + derived valence/arousal axes |

**Input:** `DATA_DIR/tweets.csv` (output of `01_eda.ipynb`)  
**Output:** `DATA_DIR/sentiment_results.csv`


## Dependencies

In [1]:
# Run once -- safe to skip if your environment already has these
!pip install vaderSentiment transformers torch tqdm


  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 55.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 69.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 70.5 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 60.5 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 71.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 65.3 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 92.6 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 81.6 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━

## Imports

In [2]:
import re
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings('ignore')


## Config

Set `DATA_DIR` to the folder produced by `01_eda.ipynb`.


In [3]:
DATA_DIR  = "/home/mhadi/Notebooks/sim3"

INPUT_CSV  = f"{DATA_DIR}/tweets.csv"
OUTPUT_CSV = f"{DATA_DIR}/sentiment_results.csv"

SENTIMENT_MODEL   = "cardiffnlp/twitter-roberta-base-sentiment-latest"
EMOTION_MODEL     = "j-hartmann/emotion-english-distilroberta-base"
TRANSFORMER_BATCH = 32

EMOTION_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]


## Load data

In [4]:
tweets = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(tweets):,} rows x {len(tweets.columns)} columns")
tweets.head()


Loaded 538 rows x 17 columns


,id,tweet,post_img,user_id,comment_to,thread_id,round,news_id,shared_from,image_id,reaction_count,day,hour,like,dislike,length,true_topics
0,1,"I'm tellin' ya, AI's gonna change everything! ...",0,56,-1,1,1,0,-1,0,15,0,0,11.0,4.0,192,['AI causing mass unemployment']
1,2,"I'm tellin' ya, AI's gonna change everything! ...",0,56,-1,2,1,0,-1,0,16,0,0,13.0,3.0,192,['AI causing mass unemployment']
2,3,"@EricThompson You're so caught up in the hype,...",0,52,2,2,2,0,-1,0,10,0,1,10.0,0.0,255,['AI causing mass unemployment']
3,4,@BrandonColeman I completely agree with you! I...,0,20,2,2,2,0,-1,0,13,0,1,13.0,0.0,363,['AI causing mass unemployment']
4,5,I'm so excited to see AI bringing an economic ...,0,19,-1,5,2,0,-1,0,10,0,1,7.0,3.0,191,['AI bringing an economic boom']


## Preprocessing

`preprocess_text` strips URLs, mentions, and HTML entities while keeping
hashtag words and (by default) emojis -- both carry sentiment signal on
social media. A `clean_text` column is added to the DataFrame; empty results
are flagged so downstream models can skip them.


In [5]:
def preprocess_text(text: str, *, keep_emojis: bool = True) -> str:
    """
    Clean a tweet for sentiment analysis.

    Steps:
      - Lowercase
      - Remove URLs
      - Remove mentions (@user)
      - Normalise hashtags (#word -> word)
      - Strip HTML entities
      - Collapse whitespace
      - Optionally keep emojis (important signal for sentiment)
    """
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)
    text = re.sub(r"&quot;", '"', text)
    text = re.sub(r"&#\d+;", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_dataframe(df: pd.DataFrame, text_col: str = "tweet") -> pd.DataFrame:
    """Add a 'clean_text' column to df."""
    print(f"[preprocess] Cleaning {len(df):,} tweets ...")
    df = df.copy()
    df["clean_text"] = df[text_col].apply(preprocess_text)
    df["is_empty"] = df["clean_text"].str.strip() == ""
    empty_n = df["is_empty"].sum()
    if empty_n:
        print(f"  Warning: {empty_n} tweets are empty after cleaning and will be skipped.")
    return df


tweets = preprocess_dataframe(tweets)


[preprocess] Cleaning 538 tweets ...


## VADER sentiment

VADER is a lexicon-based scorer -- no GPU required and runs in seconds.
It returns four scores per tweet: `neg`, `neu`, `pos` (proportions summing to 1)
and `compound` (normalised aggregate in [-1, +1]).
Label thresholds: compound >= 0.05 -> positive; <= -0.05 -> negative; else neutral.


In [6]:
def run_vader(df: pd.DataFrame, text_col: str = "clean_text") -> pd.DataFrame:
    """
    Add VADER sentiment columns:
      vader_neg, vader_neu, vader_pos, vader_compound  (raw scores)
      vader_label  (negative / neutral / positive)
    """
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    except ImportError:
        raise ImportError("pip install vaderSentiment")

    print("[vader] Scoring ...")
    analyzer = SentimentIntensityAnalyzer()
    df = df.copy()

    scores = df[text_col].apply(
        lambda t: analyzer.polarity_scores(t) if t else
                  {"neg": np.nan, "neu": np.nan, "pos": np.nan, "compound": np.nan}
    )
    scores_df = pd.DataFrame(scores.tolist(), index=df.index)
    scores_df.columns = ["vader_neg", "vader_neu", "vader_pos", "vader_compound"]
    df = pd.concat([df, scores_df], axis=1)

    def vader_label(c):
        if pd.isna(c):  return np.nan
        if c >= 0.05:   return "positive"
        if c <= -0.05:  return "negative"
        return "neutral"

    df["vader_label"] = df["vader_compound"].apply(vader_label)
    print(f"  -> done. Distribution:\n{df['vader_label'].value_counts().to_string()}\n")
    return df


tweets = run_vader(tweets)


[vader] Scoring ...
  -> done. Distribution:
vader_label
positive    332
negative    201
neutral       5



## Transformer helper

`_run_classifier` is a shared batching wrapper around any HuggingFace
`text-classification` pipeline. It handles empty strings, progress bars,
and result re-alignment so the two model-specific functions below stay lean.


In [7]:
_SENTIMENT_LABEL_MAP = {
    "label_0":  "negative", "label_1":  "neutral",  "label_2":  "positive",
    "neg":      "negative", "neu":      "neutral",   "pos":      "positive",
    "negative": "negative", "neutral":  "neutral",   "positive": "positive",
}


def _run_classifier(
    texts: list,
    model_name: str,
    batch_size: int,
    desc: str,
    device: int = -1,
) -> list:
    """
    Run a HuggingFace text-classification pipeline (top_k=None) over a list of texts.
    Empty strings receive None. Returns one result-list per input text.
    """
    from transformers import pipeline, AutoTokenizer

    print(f"[{desc}] Loading {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    clf = pipeline(
        "text-classification",
        model=model_name,
        tokenizer=tokenizer,
        top_k=None,
        truncation=True,
        max_length=128,
        device=device,
    )

    valid_mask  = [bool(t and t.strip()) for t in texts]
    valid_texts = [t for t, v in zip(texts, valid_mask) if v]

    print(f"[{desc}] Scoring {sum(valid_mask):,} non-empty texts "
          f"in batches of {batch_size} ...")
    preds = []
    for i in tqdm(range(0, len(valid_texts), batch_size), unit="batch", desc=desc):
        preds.extend(clf(valid_texts[i : i + batch_size]))

    results = [None] * len(texts)
    j = 0
    for i, valid in enumerate(valid_mask):
        if valid:
            results[i] = preds[j]
            j += 1
    return results


## RoBERTa sentiment

`cardiffnlp/twitter-roberta-base-sentiment-latest` is fine-tuned specifically
on Twitter data. It adds per-class probabilities and a compound score
(`roberta_pos - roberta_neg`) that mirrors VADER's compound for direct comparison.


In [8]:
def run_transformer_sentiment(
    df: pd.DataFrame,
    text_col: str = "clean_text",
    model_name: str = SENTIMENT_MODEL,
    batch_size: int = TRANSFORMER_BATCH,
    device: int = -1,
) -> pd.DataFrame:
    """
    Adds columns:
      roberta_neg, roberta_neu, roberta_pos  (class probabilities)
      roberta_compound = pos_prob - neg_prob, in [-1, +1]
      roberta_label    (argmax label)
      roberta_score    (argmax confidence)
    """
    results = _run_classifier(df[text_col].tolist(), model_name, batch_size,
                               "sentiment", device)

    rows = {k: [] for k in ["roberta_neg", "roberta_neu", "roberta_pos",
                             "roberta_label", "roberta_score"]}

    for res in results:
        if res is None:
            for k in rows: rows[k].append(np.nan)
            continue
        score_dict = {
            _SENTIMENT_LABEL_MAP.get(r["label"].lower(), r["label"].lower()): r["score"]
            for r in res
        }
        best = max(res, key=lambda x: x["score"])
        rows["roberta_neg"].append(score_dict.get("negative", np.nan))
        rows["roberta_neu"].append(score_dict.get("neutral",  np.nan))
        rows["roberta_pos"].append(score_dict.get("positive", np.nan))
        rows["roberta_label"].append(
            _SENTIMENT_LABEL_MAP.get(best["label"].lower(), best["label"].lower()))
        rows["roberta_score"].append(best["score"])

    df = df.copy()
    for col, vals in rows.items():
        df[col] = vals
    df["roberta_compound"] = df["roberta_pos"] - df["roberta_neg"]
    print(f"  -> Distribution:\n{df['roberta_label'].value_counts().to_string()}\n")
    return df


tweets = run_transformer_sentiment(tweets)


[sentiment] Loading cardiffnlp/twitter-roberta-base-sentiment-latest ...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 34423.06it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[sentiment] Scoring 538 non-empty texts in batches of 32 ...


sentiment: 100%|██████████| 17/17 [00:41<00:00,  2.46s/batch]

  -> Distribution:
roberta_label
negative    291
positive    163
neutral      84



## Emotion detection

`j-hartmann/emotion-english-distilroberta-base` covers Ekman's 6 basic
emotions + neutral. Two derived axes -- **valence** and **arousal** -- are
computed as weighted combinations of the raw probabilities, approximating
Russell's circumplex model and making the scores directly usable in bias
analysis downstream.


In [9]:
def run_transformer_emotion(
    df: pd.DataFrame,
    text_col: str = "clean_text",
    model_name: str = EMOTION_MODEL,
    batch_size: int = TRANSFORMER_BATCH,
    device: int = -1,
) -> pd.DataFrame:
    """
    Adds columns:
      emotion_{anger,disgust,fear,joy,neutral,sadness,surprise}  (probabilities)
      emotion_label, emotion_score  (argmax)
      emotion_valence = joy - mean(sadness, fear, disgust, anger), in [-1, +1]
      emotion_arousal = mean(anger, fear, surprise, joy)
                      - mean(sadness, disgust, neutral), in [-1, +1]
    Both axes approximate Russell's circumplex model.
    """
    results = _run_classifier(df[text_col].tolist(), model_name, batch_size,
                               "emotion", device)

    rows = {f"emotion_{e}": [] for e in EMOTION_LABELS}
    rows["emotion_label"] = []
    rows["emotion_score"] = []

    for res in results:
        if res is None:
            for k in rows: rows[k].append(np.nan)
            continue
        score_dict = {r["label"].lower(): r["score"] for r in res}
        best = max(res, key=lambda x: x["score"])
        for e in EMOTION_LABELS:
            rows[f"emotion_{e}"].append(score_dict.get(e, np.nan))
        rows["emotion_label"].append(best["label"].lower())
        rows["emotion_score"].append(best["score"])

    df = df.copy()
    for col, vals in rows.items():
        df[col] = vals

    unpleasant = df[["emotion_sadness", "emotion_fear",
                     "emotion_disgust",  "emotion_anger"]].mean(axis=1)
    df["emotion_valence"] = df["emotion_joy"] - unpleasant

    activated   = df[["emotion_anger", "emotion_fear",
                       "emotion_surprise", "emotion_joy"]].mean(axis=1)
    deactivated = df[["emotion_sadness", "emotion_disgust",
                       "emotion_neutral"]].mean(axis=1)
    df["emotion_arousal"] = activated - deactivated

    print(f"  -> Distribution:\n{df['emotion_label'].value_counts().to_string()}\n")
    return df


tweets = run_transformer_emotion(tweets)


[emotion] Loading j-hartmann/emotion-english-distilroberta-base ...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 23500.64it/s]


[emotion] Scoring 538 non-empty texts in batches of 32 ...


emotion: 100%|██████████| 17/17 [00:24<00:00,  1.43s/batch]

  -> Distribution:
emotion_label
fear        128
joy         119
surprise    106
anger        83
sadness      57
neutral      44
disgust       1



## Diagnostics

Pairwise agreement between VADER and RoBERTa gives a quick sanity check:
low agreement on a batch often signals that the preprocessing is stripping
too much signal.


In [10]:
def compute_agreement(df: pd.DataFrame) -> None:
    """Pairwise label-agreement between VADER and RoBERTa sentiment."""
    cols = [c for c in ["vader_label", "roberta_label"] if c in df.columns]
    if len(cols) < 2:
        return
    print("[agreement] Pairwise sentiment label agreement:")
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            a, b = cols[i], cols[j]
            mask = df[a].notna() & df[b].notna()
            if mask.sum() == 0:
                continue
            agree = (df.loc[mask, a] == df.loc[mask, b]).mean()
            print(f"  {a} vs {b}: {agree:.1%}  (n={mask.sum():,})")
    print()


def print_column_summary(df: pd.DataFrame) -> None:
    """Print a grouped summary of all new columns added by this notebook."""
    prefixes = ["clean_text", "is_empty", "vader_", "roberta_", "emotion_"]
    new_cols = [c for c in df.columns
                if any(c.startswith(p) or c == p for p in prefixes)]
    groups = {
        "Preprocessing":          [c for c in new_cols if c in ("clean_text", "is_empty")],
        "VADER sentiment":         [c for c in new_cols if c.startswith("vader_")],
        "RoBERTa sentiment":       [c for c in new_cols if c.startswith("roberta_")],
        "Emotion (DistilRoBERTa)": [c for c in new_cols if c.startswith("emotion_")],
    }
    print("New columns added:")
    for group, cols in groups.items():
        if cols:
            print(f"  [{group}]")
            for c in cols:
                print(f"    {c}")
    print()


compute_agreement(tweets)
print_column_summary(tweets)


[agreement] Pairwise sentiment label agreement:
  vader_label vs roberta_label: 58.4%  (n=538)

New columns added:
  [Preprocessing]
    clean_text
    is_empty
  [VADER sentiment]
    vader_neg
    vader_neu
    vader_pos
    vader_compound
    vader_label
  [RoBERTa sentiment]
    roberta_neg
    roberta_neu
    roberta_pos
    roberta_label
    roberta_score
    roberta_compound
  [Emotion (DistilRoBERTa)]
    emotion_anger
    emotion_disgust
    emotion_fear
    emotion_joy
    emotion_neutral
    emotion_sadness
    emotion_surprise
    emotion_label
    emotion_score
    emotion_valence
    emotion_arousal



## Save results

In [11]:
tweets.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(tweets):,} rows -> {OUTPUT_CSV}")


Saved 538 rows -> /home/mhadi/Notebooks/sim3/sentiment_results.csv
